# Capstone project — Air passengers, start to finish

**Submitted by: Abdullah Faisal Alsehli — academy: @SDAIAAcademy**

This notebook applies the complete course pipeline to the monthly Box & Jenkins
AirPassengers series: data checks, exploration, STL decomposition, benchmark
floors, an AutoGluon model zoo, and leakage-safe rolling-origin evaluation.

**Decision preview.** Ship AutoARIMA for this horizon. Across eight 12-month
origins its MASE is 0.647 versus 1.313 for seasonal naive, although its 80%
interval covers only 69.8% of observations and therefore needs calibration.

Full rules: `brief.md`. Grading: `rubric.md`. The notebook is Colab-ready and
runs top to bottom; the cached CSV is the same canonical AirPassengers data used
by `statsmodels.get_rdataset("AirPassengers")`.


In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    repo = Path("/content/international-airpassengers-project")
    if not repo.exists():
        !git clone -q https://github.com/abdullahf-77/international-airpassengers-project.git {repo}
    %cd /content/international-airpassengers-project
    # Install the project and AutoGluon before importing NumPy/Pandas so pip can
    # resolve a consistent environment without requiring a runtime restart.
    !pip install -q -e . autogluon.timeseries

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from coursekit import plotting as P
from coursekit import scoring

P.use_course_style()

SEASON_LENGTH = 12
H = 12
H_REPORT = 24


---
# 0. The data

Monthly international air passenger numbers, 1949-01 to 1960-12: 144
observations, no gaps. The Box & Jenkins classic (1970) — twelve years of
growth, and a year that does not repeat the same way twice.

Fetched from statsmodels' R-datasets on first run; no download step.

In [ ]:
from statsmodels.datasets import get_rdataset

# Fetch the official R-datasets copy when online. The checked-in CSV is an
# identical fallback, so a transient network error cannot break a Colab run.
try:
    raw = get_rdataset("AirPassengers").data
    t = raw["time"].astype(float).to_numpy()
    years = np.floor(t).astype(int)
    months = np.clip(np.round((t - years) * 12).astype(int) + 1, 1, 12)
    df = pd.DataFrame({
        "unique_id": "Air passengers (Box & Jenkins)",
        "ds": pd.to_datetime(pd.Series(years).astype(str) + "-" +
                             pd.Series(months).astype(str).str.zfill(2) + "-01"),
        "y": raw["value"].astype(float),
    })
except Exception as exc:
    print(f"Online fetch unavailable ({type(exc).__name__}); using the identical cached series.")
    df = pd.read_csv("air_passengers.csv", parse_dates=["ds"])

df = df.sort_values("ds").reset_index(drop=True)
df.head()


In [ ]:
expected_calendar = pd.date_range(df["ds"].min(), df["ds"].max(), freq="MS")
n_obs = len(df)
range_ = (df["ds"].min(), df["ds"].max())
n_gaps = len(expected_calendar.difference(df["ds"]))

assert n_obs == 144
assert range_[0] == pd.Timestamp("1949-01-01")
assert range_[1] == pd.Timestamp("1960-12-01")
assert n_gaps == 0
assert df["ds"].is_unique and df["y"].notna().all()

print(f"{n_obs} months, {range_[0].date()} to {range_[1].date()}, {n_gaps} gaps")


In [ ]:
P.plot_series(
    df,
    title="Passenger demand rises while the yearly swings widen",
    ylabel="passengers (thousands)",
    xlabel="",
)
plt.show()


**First read.** Passenger demand grows from roughly 100 thousand per month in
1949 to more than 400 thousand by 1960, with a repeatable summer peak and winter
trough every year. The seasonal swings widen as the level rises, so the series
is not stationary and its seasonality behaves more multiplicatively than
additively; a useful model must handle both trend and annual seasonality.


---
# 1. Exploration and shape

The Day 1 toolkit, in order. Every chart should carry its claim: a title that
says what it shows, not just "series".

**1.1 Decompose.** The STL trend carries the long rise, while the seasonal
component carries the stable within-year timing of peaks and troughs. The
remainder is much smaller than either component but becomes more volatile near
the end, consistent with the widening swings visible in the original series.


In [ ]:
from statsmodels.tsa.seasonal import STL

stl = STL(df.set_index("ds")["y"], period=SEASON_LENGTH, robust=True).fit()
dcmp = pd.DataFrame({
    "ds": df["ds"],
    "observed": df["y"],
    "trend": stl.trend.to_numpy(),
    "seasonal": stl.seasonal.to_numpy(),
    "remainder": stl.resid.to_numpy(),
})

P.decomposition_plot(
    dcmp,
    cols=["observed", "trend", "seasonal", "remainder"],
    title="STL: trend explains the rise; seasonality explains the yearly rhythm",
    ylabel="passengers (thousands)",
    x="ds",
    panel_height=1.15,
)
plt.show()


**1.2 Trend and seasonal strength.** Using the textbook STL formulas,
$F_T=1-Var(R)/Var(T+R)$ and $F_S=1-Var(R)/Var(S+R)$ (clipped at zero), both
components are extremely strong. That rules out an unseasonal level-only model
and supports models that combine trend with a 12-month seasonal mechanism.


In [ ]:
remainder = dcmp["remainder"]
trend_strength = max(0.0, 1 - remainder.var(ddof=1) /
                     (dcmp["trend"] + remainder).var(ddof=1))
seasonal_strength = max(0.0, 1 - remainder.var(ddof=1) /
                        (dcmp["seasonal"] + remainder).var(ddof=1))

print(f"trend strength: {trend_strength:.3f}")
print(f"seasonal strength: {seasonal_strength:.3f}")
assert trend_strength > 0.9 and seasonal_strength > 0.9


**1.3 Autocorrelation.** The raw series has slowly decaying positive
autocorrelation and a clear annual echo: ACF is about 0.760 at lag 12 and 0.532
at lag 24. In plain language, a strong month tends to be followed by strong
months and resembles the same month one and two years earlier—direct evidence
that both persistence and yearly seasonality belong in the model.


In [ ]:
from statsmodels.tsa.stattools import acf

P.acf_plot(
    df["y"],
    nlags=24,
    title="Raw ACF: persistent demand with annual echoes at lags 12 and 24",
    highlight_every=SEASON_LENGTH,
)
plt.show()

raw_acf = acf(df["y"], nlags=24, fft=True)
print(f"ACF lag 12: {raw_acf[12]:.3f}; lag 24: {raw_acf[24]:.3f}")


---
# 2. The floor

The seasonal naive is the first model you fit on any series - not as a
formality, as the thing everything else has to beat. Fit it, check what it
leaves on the table, and give it an honest interval.

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import SeasonalNaive, Naive

# Two explicit benchmark floors: last observation and same month last year.
benchmark_sf = StatsForecast(
    models=[Naive(), SeasonalNaive(season_length=SEASON_LENGTH)],
    freq="MS",
    n_jobs=1,
)
benchmark_fc = benchmark_sf.forecast(df=df, h=H_REPORT, level=scoring.LEVELS)
floor_fc = benchmark_fc[[c for c in benchmark_fc.columns
                         if c in {"unique_id", "ds"} or c.startswith("SeasonalNaive")]]
benchmark_fc.head()


In [ ]:
P.fan_chart(
    df,
    floor_fc.rename(columns={
        "SeasonalNaive": "mean",
        **{f"SeasonalNaive-lo-{level}": f"lo-{level}" for level in scoring.LEVELS},
        **{f"SeasonalNaive-hi-{level}": f"hi-{level}" for level in scoring.LEVELS},
    }),
    title="Seasonal naive repeats last year; uncertainty widens with horizon",
    history_tail=36,
    levels=(80, 95),
)
plt.show()


**2.1 What is left on the table.** Seasonal-naive residuals are
$y_t-y_{t-12}$. They remain mostly positive and arrive in runs; lag-1 ACF is
about 0.746, while lag-12 ACF is near zero. The floor has removed the annual
timing but not the rising level and short-run dependence. A trend-aware ARIMA,
ETS, or Theta model can target exactly that leftover structure.


In [ ]:
resid_floor = (df.set_index("ds")["y"] -
               df.set_index("ds")["y"].shift(SEASON_LENGTH)).dropna()

fig, ax = plt.subplots()
ax.axhline(0, color="0.4", lw=0.8)
ax.plot(resid_floor.index, resid_floor, color="black", lw=1.0)
ax.set(title="Seasonal-naive residuals stay positive in runs: trend remains",
       ylabel="actual minus same month last year", xlabel="")
plt.show()

P.acf_plot(
    resid_floor,
    nlags=24,
    title="Floor residual ACF: strong low-lag dependence, little lag-12 echo",
    highlight_every=SEASON_LENGTH,
)
plt.show()

floor_resid_acf = acf(resid_floor, nlags=24, fft=True)
print(f"Residual ACF lag 1: {floor_resid_acf[1]:.3f}; "
      f"lag 12: {floor_resid_acf[12]:.3f}")


---
# 3. Your model: AutoGluon

AutoGluon fits a real zoo behind one interface. In Colab the next cell installs
the time-series package automatically; locally it is skip-guarded so the
verified StatsForecast harness still runs. The final decision never relies on
AutoGluon's single holdout—it only supplies a shortlist for section 4.


In [ ]:
HAS_AG = importlib.util.find_spec("autogluon") is not None
print("AutoGluon available:", HAS_AG)
if not HAS_AG:
    print("Local verification skips section 3; Colab installs it in the setup cell.")


**3.1 The data conversion stays visible.** AutoGluon has its own
frame, its own column names, its own fit loop and its own leaderboard.
Writing it out by hand once is the point - the gap between two APIs doing
the same job is the thing to notice.

| | this course | AutoGluon |
|---|---|---|
| id column | `unique_id` | `item_id` |
| time column | `ds` | `timestamp` |
| value column | `y` | `target` (nameable) |
| frame | `pd.DataFrame` | `TimeSeriesDataFrame` |

In [ ]:
if HAS_AG:
    from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

    # The conversion is deliberately visible: no helper hides the API boundary.
    ag_long = df.rename(columns={"unique_id": "item_id", "ds": "timestamp"})
    tsdf = TimeSeriesDataFrame.from_data_frame(
        ag_long,
        id_column="item_id",
        timestamp_column="timestamp",
    )
    ag_train = tsdf.slice_by_timestep(None, -H)

    print(tsdf.head(3))
    print(f"\nfull: {len(tsdf)} rows   training before holdout: {len(ag_train)} rows")


In [ ]:
if HAS_AG:
    ZOO = {
        "Naive": {},
        "SeasonalNaive": {},
        "AutoETS": {},
        "AutoARIMA": {},
        "Theta": {},
    }

    predictor = TimeSeriesPredictor(
        prediction_length=H,
        target="y",
        eval_metric="MASE",
        freq="MS",
        verbosity=1,
    )
    predictor.fit(
        ag_train,
        hyperparameters=ZOO,
        num_val_windows=1,
        time_limit=180,
    )


In [ ]:
if HAS_AG:
    ag_leaderboard = predictor.leaderboard(tsdf, silent=True)
    display(ag_leaderboard)
    print("Models fitted:", ", ".join(ag_leaderboard["model"].astype(str)))
    print("Validation windows used by AutoGluon: 1")
else:
    print("Run this notebook in Colab to display the AutoGluon leaderboard.")


**3.2 Read what it handed back.** AutoGluon reports `score_val` as
**negative MASE**: because MASE is a loss, it flips the sign so a value closer
to zero ranks higher. The zoo contains more than one genuine model, but its
internal ranking uses only one validation window (`num_val_windows=1`). It is
useful for shortlisting AutoARIMA/Theta/ETS, not strong enough to decide what
to ship. The shortlist agrees with section 1 because these candidates model
trend plus annual seasonality; all candidates now face the same eight origins.


---
# 4. The harness

The framework's ranking came from one internal split. One holdout is not a
ranking. So the shortlist goes through the rolling-origin harness: eight
origins, twelve months each, the whole distribution scored - and the floor
stays in the table.

In [ ]:
from statsforecast.models import AutoARIMA, AutoETS, AutoTheta

CV_MODELS = [
    SeasonalNaive(season_length=SEASON_LENGTH),
    Naive(),
    AutoETS(season_length=SEASON_LENGTH),
    AutoARIMA(season_length=SEASON_LENGTH),
    AutoTheta(season_length=SEASON_LENGTH),
]

sf_cv = StatsForecast(models=CV_MODELS, freq="MS", n_jobs=1)
cv = sf_cv.cross_validation(
    df=df,
    h=H,
    step_size=H,
    n_windows=8,
    level=scoring.LEVELS,
)

assert cv["cutoff"].nunique() == 8
assert len(cv) == 8 * H
print(f"{cv['cutoff'].nunique()} origins × {H} months = {len(cv)} scored forecasts")
cv.head()


In [ ]:
MODEL_NAMES = ["SeasonalNaive", "Naive", "AutoETS", "AutoARIMA", "AutoTheta"]

RESULTS = pd.DataFrame({
    name: scoring.score_cv(cv, name, df, seasonality=SEASON_LENGTH)
    for name in MODEL_NAMES
}).T.sort_values("mase")

# Width is reported alongside empirical coverage so interval honesty is visible.
RESULTS["avg_width_80"] = [
    float((cv[f"{name}-hi-80"] - cv[f"{name}-lo-80"]).mean())
    for name in RESULTS.index
]

RESULTS.index.name = "model"
RESULTS_DISPLAY = RESULTS.copy()
for col in ["mase", "rmsse", "mase_min", "mase_max", "avg_width_80"]:
    RESULTS_DISPLAY[col] = RESULTS_DISPLAY[col].map(lambda value: f"{value:.3f}")
RESULTS_DISPLAY["crps"] = RESULTS_DISPLAY["crps"].map(lambda value: f"{value:.4f}")
RESULTS_DISPLAY["coverage_80"] = RESULTS_DISPLAY["coverage_80"].map(lambda value: f"{value:.1%}")
display(RESULTS_DISPLAY)

assert RESULTS.loc["AutoARIMA", "mase"] < RESULTS.loc["SeasonalNaive", "mase"]
assert RESULTS.loc["AutoARIMA", "crps"] < RESULTS.loc["SeasonalNaive", "crps"]


**4.1 Read the table.** The eight-origin harness confirms AutoARIMA.
It lowers MASE from **1.313** for seasonal naive to **0.647** (about a 51%
reduction) and lowers scaled CRPS from **0.0616** to **0.0339** (about a 45%
reduction). It wins both the point and distribution scores, though its fold
MASE ranges from 0.330 to 1.594, so the advantage is not uniform every year.

Intervals are the weakness. AutoARIMA's nominal 80% band covers **69.8%** of
the 96 held-out months; seasonal naive covers only **51.0%**. Both bands are
too narrow, even though AutoARIMA is markedly less under-covered. Naive has
wider coverage (68.8%) but loses badly on MASE and CRPS, so width alone is not
quality. The recommendation is AutoARIMA with interval recalibration.


### 4.2 What the selected model still misses

The selected model's one-step fitted residuals are centered near zero and show
no large recurring spikes in the first 24 ACF lags after the 12-month warm-up.
That is a substantial improvement over the floor. The remaining misses cluster
around the largest peaks and troughs and grow somewhat with the series level,
which is consistent with scale-dependent volatility and explains why the
uncalibrated interval remains too optimistic.


In [ ]:
selected_sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH)],
    freq="MS",
    n_jobs=1,
)
selected_fc = selected_sf.forecast(
    df=df,
    h=H_REPORT,
    fitted=True,
    level=scoring.LEVELS,
)
fitted = selected_sf.forecast_fitted_values()
fitted["residual"] = fitted["y"] - fitted["AutoARIMA"]
model_resid = fitted.iloc[SEASON_LENGTH:].copy()

fig, ax = plt.subplots()
ax.axhline(0, color="0.4", lw=0.8)
ax.plot(model_resid["ds"], model_resid["residual"], color="black", lw=1.0)
ax.set(title="AutoARIMA residuals: centered, with misses concentrated at extremes",
       ylabel="actual minus fitted", xlabel="")
plt.show()

P.acf_plot(
    model_resid["residual"],
    nlags=24,
    title="AutoARIMA residual ACF: little repeatable structure remains",
    highlight_every=SEASON_LENGTH,
)
plt.show()

model_acf = acf(model_resid["residual"], nlags=24, fft=True)
print(f"Residual mean: {model_resid['residual'].mean():.2f}; "
      f"lag-1 ACF: {model_acf[1]:.3f}; lag-12 ACF: {model_acf[12]:.3f}")


### 4.3 Final 24-month operational forecast

This forecast is fitted only after model selection. It illustrates the pattern
the model would ship, but all reported performance numbers above come from the
rolling-origin harness, not from this full-data fit.


In [ ]:
autoarima_fan = selected_fc.rename(columns={
    "AutoARIMA": "mean",
    **{f"AutoARIMA-lo-{level}": f"lo-{level}" for level in scoring.LEVELS},
    **{f"AutoARIMA-hi-{level}": f"hi-{level}" for level in scoring.LEVELS},
})
P.fan_chart(
    df,
    autoarima_fan,
    title="AutoARIMA: continued growth and yearly peaks, with calibrated caution",
    history_tail=48,
    levels=(80, 95),
)
plt.show()


---
# 5. The report

The manager-ready report is saved as `report.md`. Every performance number in
it comes from the same eight-origin harness above. The one-holdout AutoGluon
leaderboard is described as a shortlist only and is never used as final
evidence.


**Scope note.** This submission completes the 100-point core project on
the required AirPassengers series. It does not claim the optional external-
driver extra credit, avoiding a weak or artificial driver for a global total.
